# 📝 과제 02: 컴퓨터 기기 복합 문의 해결

## 만들 결과

가상 쇼핑몰 **모아디지털**의 사용 설명서와 FAQ를 검색해 복합 문의에 답합니다.

> KB-102를 Windows 노트북과 iPad에 번갈아 쓰려고 합니다. 두 기기를 어떻게 등록하고 전환하나요? iPad에서 일부 단축키가 다르게 동작하면 무엇을 확인해야 하나요?

기기 등록·전환과 운영체제별 단축키는 서로 다른 확인 항목입니다. 질문에 맞는 검색 방식을 고르고, 여러 질의에서 찾은 근거를 합친 뒤 원질문에 답하세요.

- Jev로 `single`, `rewrite`, `decomposition`을 선택합니다.
- 질의별 검색 순위를 RRF로 합치고 현행 원문만 선택합니다.
- 원질문 한 번 검색과 비교해 추가 질의가 실제로 보완한 근거를 설명합니다.

교안02의 다섯 노드와 함수 이름·입출력 패턴을 사용합니다. 이 과제는 문서 검색에 집중하며 가격·재고는 조회하지 않습니다.

<img src="images/shop_assignment02_overview.png" width="1050" alt="복합 상품 문의를 질의 구성, 문서 검색, 순위 융합, 원문 선택, 근거 답변으로 처리합니다.">

## 준비: 검색 자료와 제공 코드

과제01과 같은 쇼핑몰의 **문서 120개 중 현행 108개**를 검색합니다. 상품별 설정 방법, 카테고리 사용 안내, 문제 해결 FAQ와 쇼핑몰 정책을 구분하세요. 문서의 `active`가 거짓인 12개는 폐지 자료입니다. [문서 안내](data/과제/knowledge/)에서 실제 원문을 읽을 수 있습니다.

상품 120건은 별도 카탈로그이며 과제01에서 사용합니다. 이 과제의 질문은 가격·재고가 아닌 사용 방법이므로 문서 근거로 답합니다. 단축키·등록 조작은 해당 모델의 문서에서 확인하고 다른 상품의 안내를 적용하지 않습니다.

각 절의 **반환 예시**는 자료형과 데이터 구성을 보여 줍니다. 예시에 적은 입력·검색 결과를 가정하며, 모드 확률·생성 질의·답변 문장은 실제 실행마다 달라질 수 있습니다.

환경·DB·하이브리드 검색은 완성 코드로 제공합니다. 다음 순서로 진행하세요.

1. 아래 준비 셀을 실행해 문서와 검색기, 자가채점 도구를 준비합니다. 문서 임베딩은 재생성하지 않습니다.
2. 각 절의 입력·처리 순서·반환값을 읽고 학생 코드의 `...`를 완성합니다. 골격의 번호 주석은 지문의 처리 순서와 대응합니다.
3. 작성한 정의 셀 다음의 자가채점 셀을 실행합니다. 자가채점은 고정 응답을 사용하므로 외부 API를 호출하지 않습니다.
4. 6절에서 완성한 그래프를 실제 질문으로 실행합니다. Jev 판단 1회, 필요한 경우 질의 생성 1회, 구성한 질의 수만큼 검색하고, 근거가 있으면 최종 답변을 1회 생성합니다.

In [ ]:
import os
from pathlib import Path
from typing import Literal, TypedDict
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown, display
from langgraph.graph import StateGraph, START, END

# 정답 폴더에서도 같은 단원의 data를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data" / "과제"
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
llm = ChatOpenAI(model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"), use_responses_api=True)
# 배포 문서 벡터와 동일한 모델·차원으로 검색 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                  check_embedding_ctx_length=False)  # 앞 단원처럼 자동 길이 검사·분할을 끕니다.
chroma_dir = data_dir / "knowledge" / "chroma"

query_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))

In [ ]:
def evidence_id(doc):
    """문서나 상품 조회 결과에 부여한 근거 ID를 반환합니다."""
    return doc.metadata["source_id"]


vector_store = Chroma(collection_name="day54_shop_documents", persist_directory=str(chroma_dir),
                      embedding_function=embedding_model, create_collection_if_not_exists=False)
# 저장된 본문·메타데이터를 읽습니다. 문서를 다시 임베딩하지 않습니다.
stored_documents = vector_store.get(include=["documents", "metadatas"])
documents = [
    Document(id=doc_id, page_content=text, metadata={**meta, "source_type": "knowledge"})
    for doc_id, text, meta in zip(stored_documents["ids"], stored_documents["documents"],
                                 stored_documents["metadatas"], strict=True)
]
print("검색 문서:", len(documents), "/ 검색 대상:", sum(doc.metadata["active"] for doc in documents))
display(pd.DataFrame([{"ID": evidence_id(doc), "제목": doc.metadata["title"],
                       "검색 사용": doc.metadata["active"]} for doc in documents[:6]]))

In [ ]:
kiwi = Kiwi()


def kiwi_tokenize(text):
    """문서와 질문에서 명사·외국어·숫자를 같은 기준으로 추출합니다."""
    # 문서와 질문에 같은 토큰화 기준을 적용해 BM25가 단어를 비교하게 합니다.
    return [token.form.lower() for token in kiwi.tokenize(text.replace("･", "·"))
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


# 같은 현행 문서 집합의 BM25와 의미 검색을 가중 RRF로 묶습니다.
bm25 = BM25Retriever.from_documents(
    [doc for doc in documents if doc.metadata["active"]],
    preprocess_func=kiwi_tokenize, k=12,
)
dense = vector_store.as_retriever(search_kwargs={"k": 12, "filter": {"active": True}})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")


def search_documents(search_query, top_k=6):
    """하이브리드 순위에서 최대 top_k개를 반환합니다."""

    # 내부 검색은 각각 12개입니다. top_k는 결합한 뒤 전달할 수를 제한합니다.
    return hybrid.invoke(search_query)[:top_k]

In [ ]:
# [제공 코드] 자가채점은 고정 응답으로 확인하며 외부 API를 호출하지 않습니다.
from copy import deepcopy
from math import isclose
from types import SimpleNamespace
from typing import is_typeddict
from unittest.mock import Mock, patch, call

## 1. Fusion 상태를 설계하세요

**작성:** `SearchQuery`와 `FusionState` 상태 구조, `make_fusion_state` 초기화 함수를 설계하세요. 이 함수는 그래프를 실행하기 전에 새 질문의 상태를 만들며, 그래프 노드로 등록하지 않습니다.

**목적:** 원질문과 검색 상한을 보존하고, 검색·융합·답변 결과를 저장할 새 State를 만듭니다. `per_query_k`는 질의당 후보 상한(기본 3), `final_k`는 최종 원문 상한(기본 6)입니다.

**구현할 처리**

1. `SearchQuery`를 `TypedDict`로 정의합니다. `kind`는 `Literal["original", "rewrite", "subquestion"]`, `query`는 검색 문자열 `str`입니다. 하나의 항목은 검색할 질의 하나를 나타냅니다.
2. `FusionState`를 `TypedDict`로 정의합니다. 아래 표의 필드와 타입을 모두 포함하세요. 목록 안의 기록 형식은 해당 노드를 작성할 절에서 설명합니다.
3. `make_fusion_state`에서 받은 질문과 두 상한을 같은 이름의 필드에 저장합니다.
4. 나머지 필드를 표의 초기값으로 채우고 전체 상태 딕셔너리를 반환합니다. 호출할 때마다 새 목록과 딕셔너리를 만들어 서로 다른 질문의 결과가 섞이지 않게 하세요.

| 필드 | 타입 | 역할 | 초기값 |
|---|---|---|---|
| `question` | `str` | 최종 답변까지 보존할 원질문 | 입력 질문 |
| `per_query_k` | `int` | 질의 하나에서 받을 후보 수 상한 | 입력 상한, 기본 3 |
| `final_k` | `int` | 최종 답변에 전달할 원문 수 상한 | 입력 상한, 기본 6 |
| `query_mode` | `str` | Jev가 고른 검색 모드 | 빈 문자열 |
| `mode_probabilities` | `dict[str, float]` | 검색 모드별 선택 확률 | 새 빈 딕셔너리 |
| `queries` | `list[SearchQuery]` | 검색할 질의 목록 | 새 빈 목록 |
| `search_log` | `list[dict]` | 질의별 검색 순위 기록 | 새 빈 목록 |
| `fused` | `list[dict]` | 문서별 RRF 합산 점수와 순위 | 새 빈 목록 |
| `contributions` | `list[dict]` | 각 질의가 문서 점수에 기여한 기록 | 새 빈 목록 |
| `documents` | `list[Document]` | 선택한 최종 원문 | 새 빈 목록 |
| `selection_log` | `list[dict]` | 후보별 선택·제외 이유 | 새 빈 목록 |
| `draft` | `str` | 최종 답변 | 빈 문자열 |
| `evidence_ids` | `list[str]` | 답변에서 인용한 근거 ID | 새 빈 목록 |

**반환값: `dict`** — `make_fusion_state`는 표의 13개 필드를 모두 담은 초기 상태를 반환합니다. 검색 모드는 다음 노드가 선택합니다.

**반환 예시:** `make_fusion_state("KB-102 기기 전환 방법", per_query_k=2, final_k=4)`

```python
{
    "question": "KB-102 기기 전환 방법",
    "per_query_k": 2, "final_k": 4,
    "query_mode": "", "mode_probabilities": {},
    "queries": [], "search_log": [], "fused": [], "contributions": [],
    "documents": [], "selection_log": [], "draft": "", "evidence_ids": []
}
```

In [ ]:
class SearchQuery(TypedDict):
    # 1. 질의 종류와 검색 문자열의 타입을 정의하세요.
    ...


class FusionState(TypedDict):
    # 2. 표의 상태 필드와 타입을 모두 정의하세요.
    ...


def make_fusion_state(question, per_query_k=3, final_k=6):
    # 3. 입력 질문과 두 후보 상한을 저장하세요.
    # 4. 나머지 초기값을 채워 새 상태 딕셔너리를 반환하세요.
    ...

In [ ]:
state = make_fusion_state("기기 연결 문의", per_query_k=2, final_k=4)
assert is_typeddict(FusionState) and is_typeddict(SearchQuery)
assert state["question"] == "기기 연결 문의" and state["per_query_k"] == 2 and state["final_k"] == 4
assert state["query_mode"] == "" and state["mode_probabilities"] == {}
state["queries"].append({"kind": "original", "query": "확인"})
assert make_fusion_state("다른 문의")["queries"] == []
print("Fusion 초기 상태 확인 완료")

## 2. 모드에 맞게 질의를 구성하세요

| 모드 | 검색할 질의 목록 | 각 항목의 `kind` |
|---|---|---|
| `single` | 원질문 1개 | `original` |
| `rewrite` | 원질문을 먼저 담고 재작성 질의들을 뒤에 추가 | 원질문은 `original`, 추가 질의는 `rewrite` |
| `decomposition` | 서로 다른 확인 항목의 하위 질문들만 포함 | 모두 `subquestion` |

`Choice` 판단 기준과 `QueryVariants`·질의 생성 프롬프트는 완성 코드로 제공합니다. 생성 질의 상한은 5개이며, 필요한 만큼만 만듭니다. `rewrite`는 원질문을 포함해 최대 6개, `decomposition`은 최대 5개를 검색합니다. 어떤 모드에서도 원질문은 State에 그대로 보존해 최종 답변에 사용합니다.

<img src="images/node_flow/shop_fusion_expand_queries.png" width="1050" alt="과제 전체 fusion 흐름에서 expand_queries 노드만 강조합니다.">

In [ ]:
mode_question = Choice(
    instructions=(
        "질문에 필요한 검색 질의 구성을 하나 고르세요. "
        "독립적인 정보 요구가 여러 개이면 decomposition을 우선합니다. "
        "하나의 정보 요구이면 표현을 정리할 필요가 있는지 보고 rewrite 또는 single을 고르세요. "
        "수량·시점·대상 같은 조건이 여러 개라는 이유만으로 질문을 분해하지 마세요. "
        "검색 결과를 보지 않은 단계이므로 문서 존재 여부나 검색 성공을 추측하지 마세요."
    ),
    criteria={
        "single": "하나의 정보 요구이며 대상과 검색할 내용이 구체적이어서 원질문을 그대로 검색하면 되는 질문",
        "rewrite": "하나의 정보 요구이지만 구어적·장황한 표현을 핵심 검색어로 정리하면 도움이 되는 질문",
        "decomposition": "서로 독립적으로 확인할 정보 요구가 둘 이상이어서 각각 검색한 근거를 함께 모아야 하는 질문",
    },
)


class QueryVariants(BaseModel):
    queries: list[str] = Field(max_length=5, description=(
        "선택된 모드에 맞게 만든 검색 질의 목록입니다. "
        "rewrite이면 같은 정보 요구를 다른 용어로 표현하고, "
        "decomposition이면 서로 다른 확인 항목을 하위 질문으로 나눕니다. "
        "각 질의에 필요한 대상·수량·시점·조건을 남기고, 중복 없이 최대 5개 안에서 필요한 만큼 작성합니다."
    ))

query_prompt = ChatPromptTemplate.from_messages([
    ("system", "검색 모드는 {mode}입니다. 결과는 queries에 넣으세요. "
     "rewrite이면 원질문의 전체 의미를 유지한 다른 검색 표현들을 만드세요. 원질문 자체는 다시 넣지 마세요. "
     "decomposition이면 원질문에 직접 요청된 확인 항목만 분리하고, 각 질문을 독립적으로 검색할 수 있게 작성하세요. "
     "하나의 확인 항목을 비슷한 질문 여러 개로 반복하지 마세요. "
     "도움이 될 것 같은 추가 확인이나 후속 업무는 새 질의로 만들지 마세요. "
     "질문에 없는 서류명·준비물·담당자·기한 등을 예상 답처럼 질의에 넣지 마세요. "
     "질의는 최대 5개이며, 5개를 억지로 채우지 마세요. 확인 항목이 더 많으면 관련 항목을 묶되 누락하지 마세요. "
     "어순만 바꾸거나 같은 항목을 반복해 개수를 늘리지 마세요. "
     "원질문에 명시된 조건과 선후 관계를 보존하고, 없는 요구나 추측한 답변은 추가하지 마세요."),
    ("human", "{question}"),
])
query_chain = query_prompt | llm.with_structured_output(QueryVariants)

**작성:** `expand_queries(state)` 노드를 설계하세요.

**목적:** Jev로 검색 모드를 선택하고, 그 모드에 맞는 검색 질의 목록을 만듭니다.

**구현할 처리**
1. `query_classifier.invoke`에 dict 하나를 전달합니다. `state` 키에는 원질문 문자열, `questions` 키에는 `query_mode`와 `mode_question`을 연결한 dict를 넣습니다.
2. 응답의 `choices["query_mode"]`에서 선택된 `choice`와 모드별 `probabilities`를 확인합니다.
3. `single`이면 원질문 한 개만 사용하고 질의 생성 LLM은 호출하지 않습니다.
4. 다른 모드는 원질문 `question`과 선택한 `mode`로 `query_chain`을 한 번 호출합니다. `rewrite`는 원질문 뒤에 재작성 질의를, `decomposition`은 하위 질문만 담습니다.
5. 원질문과 입력 State를 보존하고 아래 dict를 반환합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `query_mode` | Jev가 선택한 모드 문자열 |
| `mode_probabilities` | Jev가 반환한 모드별 확률 dict. 검색 성공 확률은 아님 |
| `queries` | `kind`와 `query`를 담은 dict 목록. `kind`는 원질문에 `original`, 재작성에 `rewrite`, 하위 질문에 `subquestion` 사용 |

**반환 예시:** “KB-102의 기기 등록 방법과 iPad 단축키 확인 사항을 알려 주세요.”에 Jev가 `decomposition`을 선택하고 질의 두 개를 생성한 경우입니다. 확률은 설명용 예시입니다.

```python
{
    "query_mode": "decomposition",
    "mode_probabilities": {"single": 0.1, "rewrite": 0.2, "decomposition": 0.7},
    "queries": [
        {"kind": "subquestion", "query": "KB-102 기기 등록 방법"},
        {"kind": "subquestion", "query": "KB-102 iPad 단축키 확인 사항"}
    ]
}
```

In [ ]:
# [노드] 검색 표현 준비
def expand_queries(state: FusionState):
    """모드에 맞는 검색 질의와 Jev의 판단 결과를 반환합니다."""
    # 1. 원질문과 판단 항목을 담은 단일 딕셔너리로 Jev를 호출하세요.
    # 2. 선택된 모드와 모드별 확률을 읽으세요.
    # 3. single이면 원질문 하나를 검색 목록으로 준비하세요.
    # 4. 나머지 모드는 생성 질의를 받아 모드별 목록을 구성하세요.
    # 5. query_mode·mode_probabilities·queries를 담은 dict를 반환하세요.
    ...

In [ ]:
# 세 모드의 질의 구성과 single의 LLM 호출 생략을 고정 응답으로 확인합니다.
state = make_fusion_state("등록 방법과 단축키 설정")
for mode in ["single", "rewrite", "decomposition"]:
    decision = SimpleNamespace(choices={"query_mode": SimpleNamespace(choice=mode, probabilities={mode: 1.0})})
    generated = Mock(invoke=Mock(return_value=SimpleNamespace(queries=["기기 등록", "단축키 설정"])))
    before = deepcopy(state)
    with patch.dict(globals(), {"query_classifier": Mock(invoke=Mock(return_value=decision)), "query_chain": generated}):
        expanded = expand_queries(state)
        assert expanded["query_mode"] == mode and state == before
        assert expanded["mode_probabilities"] == {mode: 1.0}
        query_classifier.invoke.assert_called_once_with({"state": state["question"], "questions": {"query_mode": mode_question}})
        expected_texts = {"single": [state["question"]], "rewrite": [state["question"], "기기 등록", "단축키 설정"], "decomposition": ["기기 등록", "단축키 설정"]}
        assert [item["query"] for item in expanded["queries"]] == expected_texts[mode]
        if mode != "single":
            generated.invoke.assert_called_once_with({"question": state["question"], "mode": mode})
        assert generated.invoke.call_count == (0 if mode == "single" else 1)
        kinds = [item["kind"] for item in expanded["queries"]]
        assert kinds == ({"single": ["original"], "rewrite": ["original", "rewrite", "rewrite"], "decomposition": ["subquestion", "subquestion"]}[mode])
print("검색 모드별 질의 구성 확인 완료")

## 3. 질의별 검색 순위를 남기세요

<img src="images/node_flow/shop_fusion_search_many.png" width="1050" alt="과제 전체 fusion 흐름에서 search_many 노드만 강조합니다.">

**작성:** `search_many(state)` 노드를 설계하세요.

**목적:** 각 검색 질의의 결과와 순위를 별도로 기록합니다.

**구현할 처리**
1. 새 검색 기록 목록을 준비합니다.
2. `queries`의 순서대로 `search_documents`를 한 번씩 호출합니다. 검색어는 각 항목의 `query`, `top_k`는 `per_query_k`입니다.
3. 질의마다 아래 형식으로 기록합니다. 결과가 없으면 `candidate_ids=[]`로 기록을 남깁니다.
4. 입력 State를 수정하지 않고 질의 순서대로 모은 기록을 반환합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `search_log` | 질의 순서대로 모은 검색 기록 목록 |

각 검색 기록은 아래 키를 담은 dict입니다.

| 기록의 키 | 값 |
|---|---|
| `source` | `'knowledge'` |
| `kind`, `query` | 검색에 사용한 질의의 종류와 문자열 |
| `candidate_ids` | 반환 문서에 `evidence_id`를 적용한 ID 목록. 검색 순서 유지 |

**반환 예시:** 두 하위 질의를 검색해 첫 질의에서 문서 두 개, 둘째 질의에서 문서 한 개를 받은 경우입니다. 문서 본문은 여기서 반환하지 않습니다.

```python
{
    "search_log": [
        {"source": "knowledge", "kind": "subquestion",
         "query": "KB-102 기기 등록 방법",
         "candidate_ids": ["guide:KB-102-pairing", "guide:KB-102-shortcuts"]},
        {"source": "knowledge", "kind": "subquestion",
         "query": "KB-102 iPad 단축키 확인 사항",
         "candidate_ids": ["guide:KB-102-shortcuts"]}
    ]
}
```

In [ ]:
# [노드] 질의별 업무 지침 검색
def search_many(state: FusionState):
    # 1. 이번 질의들의 검색 기록을 담을 목록을 준비하세요.
    # 2. 각 질의로 검색하고 질의당 후보 상한을 전달하세요.
    # 3. 빈 결과도 포함해 질의 정보와 순위대로 된 근거 ID를 기록하세요.
    # 4. 검색 기록 목록을 search_log에 담은 dict를 반환하세요.
    ...

In [ ]:
state = make_fusion_state("확인", per_query_k=2)
state["queries"] = [{"kind": "original", "query": "첫 질의"}, {"kind": "rewrite", "query": "둘째 질의"}]
doc = Document(page_content="근거", metadata={"source_id": "check:A"})
mock_search = Mock(side_effect=[[doc], []])
before = deepcopy(state)
with patch.dict(globals(), {"search_documents": mock_search}):
    searched = search_many(state)
    assert mock_search.call_count == 2 and len(searched["search_log"]) == 2
    for invoked, query in zip(mock_search.call_args_list, ["첫 질의", "둘째 질의"]):
        assert (invoked.args[0] if invoked.args else invoked.kwargs["search_query"]) == query
        assert (invoked.kwargs["top_k"] if "top_k" in invoked.kwargs else invoked.args[1]) == 2
    assert searched["search_log"][0]["candidate_ids"] == ["check:A"]
    assert searched["search_log"][1]["candidate_ids"] == [] and state == before
    assert [(r["source"], r["kind"], r["query"]) for r in searched["search_log"]] == [("knowledge", "original", "첫 질의"), ("knowledge", "rewrite", "둘째 질의")]
print("질의별 순위 기록 확인 완료")

## 4. RRF 점수와 질의별 기여를 계산하세요

<img src="images/node_flow/shop_fusion_fuse.png" width="1050" alt="과제 전체 fusion 흐름에서 fuse 노드만 강조합니다.">

**작성:** `fuse(state)` 노드를 설계하세요.

**목적:** 질의별 검색 순위를 RRF로 합산하고, 문서 점수에 대한 각 질의의 기여를 남깁니다.

**구현할 처리**
1. 문서별 누적 점수와 기여 기록을 준비합니다.
2. `search_log`의 각 `candidate_ids`에서 첫 등장 순서로 중복을 제거하고 1부터 등수를 매깁니다. 다른 질의에서 같은 문서가 나온 것은 별도로 계산합니다.
3. `1 / (60 + rank)`를 해당 문서의 누적 점수에 더하고 질의별 기여를 기록합니다.
4. 누적 점수 내림차순, 동점이면 문서 ID 오름차순으로 정렬합니다.
5. 입력 State를 수정하지 않고 아래 dict를 반환합니다. 현행 원문 선택은 다음 단계에서 처리합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `fused` | 정렬된 문서별 점수 목록. 각 항목은 `doc_id`와 합산 점수 `score`를 가진 dict |
| `contributions` | 질의별 기여 목록. 각 항목은 `doc_id`, `source`, `kind`, `query`, `rank`, `contribution`을 가진 dict |

`rank`는 해당 질의의 중복 제거 후 등수이고, `contribution`은 그 질의에서 더한 RRF 점수입니다.

**반환 예시:** 3절의 검색 기록을 합친 결과입니다. `shortcuts` 문서는 두 질의의 2위·1위 점수를 합쳐 최종 1위가 됩니다. 아래 소수는 표시를 위해 반올림했으며, 계산할 때는 반올림하지 않습니다.

```python
{
    "fused": [
        {"doc_id": "guide:KB-102-shortcuts", "score": 0.032522475},
        {"doc_id": "guide:KB-102-pairing", "score": 0.016393443}
    ],
    "contributions": [
        {"doc_id": "guide:KB-102-pairing", "source": "knowledge",
         "kind": "subquestion", "query": "KB-102 기기 등록 방법",
         "rank": 1, "contribution": 0.016393443},
        {"doc_id": "guide:KB-102-shortcuts", "source": "knowledge",
         "kind": "subquestion", "query": "KB-102 기기 등록 방법",
         "rank": 2, "contribution": 0.016129032},
        {"doc_id": "guide:KB-102-shortcuts", "source": "knowledge",
         "kind": "subquestion", "query": "KB-102 iPad 단축키 확인 사항",
         "rank": 1, "contribution": 0.016393443}
    ]
}
```

In [ ]:
# [노드] 검색 순위 융합
def fuse(state: FusionState):
    # 1. 문서별 누적 점수와 기여 기록을 준비하세요.
    # 2. 각 검색 목록 안의 중복을 제거한 뒤 등수를 매기세요.
    # 3. 등수별 기여 점수를 더하고 그 출처를 기록하세요.
    # 4. 점수 내림차순, 동점 ID 오름차순으로 합산 순위를 정렬하세요.
    # 5. fused·contributions를 담은 dict를 반환하세요.
    ...

In [ ]:
state = make_fusion_state("고정 순위 확인")
state["search_log"] = [
    {"source": "knowledge", "kind": "original", "query": "등록", "candidate_ids": ["A", "A", "B"]},
    {"source": "knowledge", "kind": "subquestion", "query": "전환", "candidate_ids": ["B", "A"]},
]
before = deepcopy(state)
rrf_result = fuse(state)
assert [r["doc_id"] for r in rrf_result["fused"]] == ["A", "B"]
assert all(isclose(r["score"], 1/61 + 1/62) for r in rrf_result["fused"])
assert len(rrf_result["contributions"]) == 4 and state == before
assert sorted((r["doc_id"], r["query"], r["rank"]) for r in rrf_result["contributions"]) == sorted([("A", "등록", 1), ("B", "등록", 2), ("B", "전환", 1), ("A", "전환", 2)])
assert all(isclose(r["contribution"], 1/(60+r["rank"])) for r in rrf_result["contributions"])
assert sorted((r["source"], r["kind"]) for r in rrf_result["contributions"]) == [("knowledge", "original")]*2 + [("knowledge", "subquestion")]*2
print("중복 ID·동점·기여 점수 확인 완료")

## 5. 현재 사용할 수 있는 원문을 선택하세요

<img src="images/node_flow/shop_fusion_select_context.png" width="1050" alt="과제 전체 fusion 흐름에서 select_context 노드만 강조합니다.">

In [ ]:
document_by_id = {evidence_id(doc): doc for doc in documents}

**작성:** `select_documents(fused, document_index, final_k)` 보조 함수와 `select_context(state)` 노드를 설계하세요.

**`select_documents(fused, document_index, final_k)`: 원문 선택 보조 함수**

**목적:** 융합 순위에서 현행 원문을 최대 `final_k`개 선택합니다.

**구현할 처리**

1. 선택 문서와 후보별 기록 목록을 준비합니다.
2. `fused` 순서대로 `doc_id`에 해당하는 원문을 `document_index`에서 찾습니다.
3. 아래 표에서 처음 해당하는 이유를 적용하고, `선택`인 문서만 추가합니다. 본문·메타데이터는 유지합니다.
4. 상한에 도달한 뒤에도 모든 후보의 `doc_id`와 `reason`을 기록합니다.
5. 아래의 두 키를 담은 dict를 반환합니다.

| 판단 순서 | 조건 | `reason` 값 |
|---|---|---|
| 1 | 원문 없음 | `'원문 없음'` |
| 2 | `metadata['active']`가 거짓 | `'폐지 문서'` |
| 3 | 이미 `final_k`개 선택 | `'최종 개수 한도'` |
| 4 | 나머지: 원문 추가 | `'선택'` |

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `documents` | 융합 순서대로 선택한 현행 `Document` 목록. 최대 `final_k`개 |
| `selection_log` | 모든 후보의 `doc_id`와 `reason`을 담은 dict 목록. 후보 순서 유지 |

**`select_context(state)`: 최종 근거 선택 노드**

**목적:** 보조 함수의 원문 선택 결과를 그래프에 전달합니다.

**구현할 처리**

1. `state['fused']`, ID별 원문 사전 `document_by_id`, `state['final_k']`를 차례로 `select_documents`에 전달합니다.
2. 보조 함수의 결과를 그대로 반환합니다. 그래프에는 이 노드만 등록합니다.

**반환값: `dict`** — `documents`와 `selection_log` 두 키를 담은 보조 함수의 결과입니다.

**반환 예시:** 4절의 두 문서가 모두 현행이고 `final_k=1`인 경우입니다. `shortcuts_doc`는 `guide:KB-102-shortcuts`의 본문·메타데이터를 가진 Document 객체입니다. 보조 함수와 노드는 같은 형식의 dict를 반환합니다.

```python
{
    "documents": [shortcuts_doc],
    "selection_log": [
        {"doc_id": "guide:KB-102-shortcuts", "reason": "선택"},
        {"doc_id": "guide:KB-102-pairing", "reason": "최종 개수 한도"}
    ]
}
```

In [ ]:
# [보조 함수] 융합 순위에서 원문 선택
def select_documents(fused, document_index, final_k):
    # 1. 선택 원문과 후보별 기록을 담을 목록을 준비하세요.
    # 2. 융합 순서대로 후보 ID에 해당하는 원문을 찾으세요.
    # 3. 표의 우선순위로 이유를 정하고 사용할 원문만 선택하세요.
    # 4. 상한 이후 후보까지 선택·제외 이유를 모두 기록하세요.
    # 5. documents·selection_log를 담은 dict를 반환하세요.
    ...


# [노드] 최종 근거 선택
def select_context(state: FusionState):
    # 1. 융합 순위·원문 목록·최종 상한을 보조 함수에 전달하세요.
    # 2. documents·selection_log를 가진 결과 dict를 그대로 반환하세요.
    ...

In [ ]:
# 폐지 문서와 없는 원문이 높게 배치돼도 최종 답변에 전달하지 않습니다.
ranking = [{"doc_id": key, "score": 1/(60+i)} for i,key in enumerate(
    ["archive:keyboard-v1", "missing:guide", "guide:KB-102-pairing", "guide:KB-102-shortcuts"], start=1)]
selected = select_documents(ranking, document_by_id, final_k=1)
assert [evidence_id(d) for d in selected["documents"]] == ["guide:KB-102-pairing"]
assert [r["reason"] for r in selected["selection_log"]] == ["폐지 문서", "원문 없음", "선택", "최종 개수 한도"]
state = make_fusion_state("원문 선택", final_k=1)
state["fused"] = ranking
assert select_context(state) == selected
assert all(d.metadata["active"] for d in selected["documents"])
print("현행 원문과 최종 상한 확인 완료")

## 6. 답변까지 연결하고 추가 검색의 가치를 확인하세요

**제공:** `generate(state)` 답변 생성 노드는 완성 코드입니다.

**목적:** 선택한 원문을 근거로 원질문에 답변합니다.

**처리:** `question`과 `documents`로 답변을 생성합니다. 근거가 없으면 모델 호출 없이 안내합니다. 검색용 하위 질문으로 원질문을 바꾸지 않습니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `draft` | 최종 답변 문자열. 근거가 없으면 근거를 찾지 못했다는 안내 |
| `evidence_ids` | 답변에서 인용한 근거 ID 문자열 목록. 근거가 없으면 `[]` |

**반환 예시:** “KB-102의 iPad 단축키가 다르게 동작하면 무엇을 확인하나요?”라는 원질문에 선택한 단축키 문서로 답한 경우입니다.

```python
{
    "draft": "iPadOS용 OS 모드와 보조 키 설정, 앱의 단축키 지원 여부를 확인하세요. [guide:KB-102-shortcuts]",
    "evidence_ids": ["guide:KB-102-shortcuts"]
}
```

아래 제공 셀을 실행한 뒤 그래프를 연결하세요.

<img src="images/node_flow/shop_fusion_generate.png" width="1050" alt="과제 전체 fusion 흐름에서 generate 노드만 강조합니다.">

In [ ]:
class GroundedAnswer(BaseModel):
    answer: str = Field(description="원질문에 대한 답변입니다. 확인된 사실마다 [근거 ID]를 붙이고 확인되지 않은 항목을 구분합니다.")
    evidence_ids: list[str] = Field(description="답변에서 실제로 인용한 제공 근거 ID 목록입니다. 없는 ID를 만들지 않습니다.")


# [보조 함수·제공] 근거 ID·출처·본문을 답변 모델이 읽을 문자열로 만듭니다.
def format_documents(docs):
    return "\n\n".join(f"[{evidence_id(doc)}] {doc.metadata['title']}\n출처: {doc.metadata['source']}\n{doc.page_content}" for doc in docs)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 근거로만 원질문에 답하세요. 사실에는 [근거 ID]를 붙이고 실제 사용한 ID를 evidence_ids에 넣으세요. "
     "상품번호별 가격·재고·기능을 섞지 마세요. 상품 DB의 사실과 일반 사용 안내의 조건을 구분하세요. "
     "다른 상품의 설정키나 지원 기능을 문의한 상품에 적용하지 마세요. 일반 안내는 그 기능을 지원하는 제품에만 적용됩니다. "
     "Bluetooth 기기 등록 대수, 한 대씩 전환, 동시 입력은 서로 다릅니다. USB-C 단자만 보고 영상·충전·데이터 기능을 확정하지 마세요. "
     "원질문에 요청된 항목을 빠뜨리지 말고 근거가 없는 항목은 확인 불가로 밝혀 주세요. "
     "가격·재고는 catalog 근거에 있을 때만 답하고 실시간 재고라고 표현하지 마세요. "
     "조건 검색은 가격순 최대6개이므로 전체 상품이나 유일한 선택지라고 단정하지 마세요. "
     "조회되지 않은 상품 안내는 마지막 별도 문단에 적고 인용을 붙이지 마세요. "
     "조회 안내 문장은 사실 인용 규칙의 예외입니다. 다른 상품의 catalog ID로 미조회 사실을 뒷받침하지 마세요. "
     "지원 여부가 조건부이면 첫 문장부터 조건을 설명하고, 구매·배송·수리·호환을 확정 약속하지 마세요. "
     "간결하게 답하며 자료에 없는 가격·도착일·입고일·버튼 조합을 만들지 마세요."),
    ("human", "질문: {question}\n인용 가능한 ID: {allowed_ids}\n근거:\n{context}"),
])
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)

In [ ]:
# [노드] 상품 사실과 안내 근거로 답변
def generate(state):
    # 1. 근거가 없으면 답변 모델을 호출하지 않고 안내합니다.
    if not state["documents"]:
        return {"draft": "제공 자료에서 답변 근거를 찾지 못했습니다.", "evidence_ids": []}
    # 2. 실제 제공하는 문서에서만 인용 가능한 ID를 모읍니다.
    allowed_ids = [evidence_id(doc) for doc in state["documents"]]
    # 3. 검색어가 아닌 원질문과 준비된 근거·조회 안내로 답변을 생성합니다.
    answer = answer_chain.invoke({"question": state["question"], "allowed_ids": allowed_ids,
        "context": format_documents(state["documents"]) + "\n조회 안내: " + state.get("product_notice", "")})
    # 4. 모델 응답의 필드명을 State의 답변 필드명에 맞춰 반환합니다.
    return {"draft": answer.answer, "evidence_ids": answer.evidence_ids}

**작성:** 앞에서 완성한 노드들을 연결하는 그래프 구성 코드를 설계하세요. 이 셀은 별도의 노드 함수가 아니라, 각 노드를 등록하고 실행 순서를 정하는 코드입니다.

**목적:** 앞에서 만든 다섯 노드를 순서대로 실행할 `fusion_app`을 만듭니다.

**구현할 처리**

1. `FusionState`를 사용하는 `StateGraph`를 만들고 `builder`에 저장합니다.
2. 다섯 함수를 각각 같은 이름의 노드로 등록합니다. `select_documents`와 `make_fusion_state`는 보조·초기화 함수이므로 등록하지 않습니다.
3. `START`에서 시작해 `expand_queries` → `search_many` → `fuse` → `select_context` → `generate` → `END` 순서로 연결합니다. 모드별 처리는 `expand_queries` 내부에서 끝나므로 여기에는 별도의 라우팅 함수나 조건부 엣지가 필요하지 않습니다.
4. 연결한 그래프를 컴파일해 `fusion_app`에 저장합니다. 컴파일은 실행 준비이며, 실제 검색·답변은 다음 셀의 `invoke`에서 시작합니다.
5. 제공된 표시 코드로 실제 그래프를 열어 위 그림과 노드 이름·순서를 비교합니다.

**만들 결과:** 다음 실행 셀에서 사용할 컴파일된 그래프 `fusion_app`입니다. 이 구성 셀은 노드의 상태 갱신 딕셔너리를 반환하는 곳이 아닙니다.

In [ ]:
# 1. FusionState를 공유하는 그래프를 준비하세요.
builder = ...
# 2. 다섯 노드를 등록하세요.
# 3. 시작부터 종료까지 지문 순서대로 연결하세요.
...
# 4. 컴파일한 그래프를 저장하세요.
fusion_app = ...
# 5. 실제 그래프를 표시해 정적 흐름 그림과 비교합니다.
display(Image(fusion_app.get_graph().draw_mermaid_png()))

In [ ]:
question = "KB-102를 Windows 노트북과 iPad에 번갈아 쓰려고 합니다. 두 기기를 어떻게 등록하고 전환하나요? iPad에서 일부 단축키가 다르게 동작하면 무엇을 확인해야 하나요?"
fusion_result = fusion_app.invoke(make_fusion_state(question))
print("Jev 선택:", fusion_result["query_mode"])
display(pd.DataFrame(fusion_result["queries"]))
display(pd.DataFrame(fusion_result["contributions"]))
display(pd.DataFrame(fusion_result["selection_log"]))
display(Markdown(fusion_result["draft"]))

**제공 코드 실행:** 같은 질문과 후보 수 상한으로 원질문 검색과 비교하세요. 아래 비교 코드는 완성되어 있으며, 방금 작성한 검색·융합·선택 함수를 재사용합니다.

1. 위 실행 결과에서 Jev가 고른 모드, 실제 질의, 선택 원문과 최종 답변을 먼저 읽습니다.
2. 아래 셀을 실행합니다. 비교 기준은 Jev·질의 생성 없이 원질문 한 개만 검색하고, 두 방법은 같은 `per_query_k`, `final_k`를 사용합니다. 기준 답변은 다시 생성하지 않습니다.
3. 비교 표에서 각 문서가 어느 방법으로 선택됐는지 확인하고, 마지막에 표시한 검색 횟수도 비교합니다. 자동 선택이 `single`이면 두 방법의 차이가 없을 수 있습니다.

In [ ]:
# [제공 코드] 원질문 검색의 근거와 실제 자동 선택 결과를 비교합니다.
single_result = make_fusion_state(question)
single_result["queries"] = [{"kind": "original", "query": question}]
single_result.update(search_many(single_result))
single_result.update(fuse(single_result))
single_result.update(select_context(single_result))
single_ids = {evidence_id(doc) for doc in single_result["documents"]}
fusion_ids = {evidence_id(doc) for doc in fusion_result["documents"]}
display(pd.DataFrame([
    {"ID": doc_id, "제목": document_by_id[doc_id].metadata["title"],
     "원질문 검색": doc_id in single_ids, "자동 선택 검색": doc_id in fusion_ids}
    for doc_id in sorted(single_ids | fusion_ids)
]))
print("검색 횟수:", len(single_result["search_log"]), "→", len(fusion_result["search_log"]))

## 결과를 읽고 점검하세요

1. **질문 충족:** 최종 답변이 Windows 노트북·iPad의 등록과 전환, iPad 단축키 점검을 모두 다루는지 확인합니다. 근거가 없는 항목은 확인 불가로 구분해야 합니다.
2. **사실 확인:** `guide:KB-102-pairing`과 `guide:KB-102-shortcuts` 등 실제 선택 원문을 열고 버튼 조합·등록 대수·OS 설정 안내를 대조합니다. 다른 모델의 설정을 KB-102에 적용하거나, 여러 기기 등록을 동시 입력으로 설명하지 않아야 합니다.
3. **인용 확인:** 답변 본문의 근거 ID와 `evidence_ids`를 비교하고, 둘 모두 실제 제공한 원문에 포함되는지 확인합니다.
4. **추가 검색의 가치:** 비교 표와 `contributions`를 함께 읽어 추가 질의가 어떤 문서를 찾거나 순위를 높였는지 설명합니다. 그 문서가 원질문에 답하는 데 필요한 근거인지도 확인하세요. 문서 수 증가만으로 개선을 단정하지 않습니다.
5. **결론 작성:** 필요한 근거가 빠졌거나 원질문 검색과 차이가 없으면 그대로 적습니다. 개선된 근거와 추가 검색 횟수를 함께 근거로 삼으세요. 모델의 문장·모드·순위 자체를 고정 정답으로 채점하지 않습니다.